In [ ]:
# Run this once at the beginning of your notebook
%load_ext autoreload
%autoreload 2

In [ ]:
# Set up everything ready for analysis
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
import pickle
import os
## plumed (i may or may not use it-cool to have though!)
import plumed
# MDAnalysis for trajectory handling
import MDAnalysis as mda
from MDAnalysis.analysis.base import AnalysisBase, Results

# Progress bar
from tqdm import tqdm

# Import your custom class (the path to some sheise may change, you may get error msgs)
import sys
sys.path.append('/Users/roev0007/Documents/solvation_shells/solvation_shells')  # For utils
sys.path.append('/Users/roev0007/Documents/solvation_shells/my_scripts')  # For scripts
from MolecularAnalysis import MolecularAnalysis
from MolecularAnalysisPlotter import MolecularAnalysisPlotter
from ZDirectionalAnalysis import ZDirectionalAnalysis
from ZDirectionalPlotter import ZDirectionalPlotter


In [ ]:
## 
# Check total frames
u = mda.Universe('nvt.tpr', 'nvt.trr')
total_frames = len(u.trajectory)
print(f"Total frames available: {total_frames}")

In [ ]:
oc_analysis = MolecularAnalysis(
    top='nvt.tpr',
    traj='nvt.trr',
    solute_sel='resname api',  # Your CIP molecule
    solvent_sel='resname SOL or resname WAT',
    cation_sel='name NA or name Na or name CA or name Ca',
    anion_sel='name CL',
    center_method='COM'
)

In [ ]:
# Recreate plotter to load updated code
import importlib
import sys

# Force complete reload of the module
module_name = 'MolecularAnalysisPlotter'
if module_name in sys.modules:
    del sys.modules[module_name]

# Also clear any cached imports
for key in list(sys.modules.keys()):
    if 'MolecularAnalysisPlotter' in key:
        del sys.modules[key]

# Now import fresh
from MolecularAnalysisPlotter import MolecularAnalysisPlotter
# Recreate plotter
plotter = MolecularAnalysisPlotter(oc_analysis)
print("✓ Plotter recreated with FIXED image export code")
print("✓ Resolution is now set by creating a new high-res view")

In [ ]:
oc_analysis.define_selections({
    'CIP_parts': {
        'quinolone': 'resname api and (name N1 or name C or name C2 or name C3 or name C7 or name C8 or name C9 or name C14 or name C15 or name C16)',
        'piperazine': 'resname api and (name N or name N2 or name C10 or name C11 or name C12 or name C13)',
        'carboxylic_acid':'resname api and (name O1 or name O2 or name C1)',
        'cyclopropyl': 'resname api and (name C4 or name C5 or name C6)',
        'O_ketone': 'resname api and name O',
        'fluoride': 'resname api and name F'
    },

    'solvent': {
        'water_oxygen': 'resname SOL WAT and (name OW or name Ow)',
        'water_hydrogen': 'resname SOL WAT and (name HW1 or name HW2 or name Hw1 or name Hw2)'
    },

    'MMT_surface': {
        'surface_oxygen':'resname MMT and name Ob',
        'surface_silicon': 'resname MMT and name Si',
        'octahedral_mg': 'resname MMT and (name Mgo or name MGO)'
    }
})

In [ ]:
## explicitly define varibales
quinolone = oc_analysis.sel('quinolone')
carboxylic_acid = oc_analysis.sel('carboxylic_acid')
piperazine = oc_analysis.sel('piperazine')
cyclopropyl = oc_analysis.sel('cyclopropyl')
water_oxygen = oc_analysis.sel('water_oxygen')

In [ ]:
# # RDF analysis
# # Run this cell if you did not define varibales explicitly
# rdf_water = oc_analysis.molecular_rdf(
#     bin_width=0.1,
#     group1_sel=[
#         oc_analysis.sel('quinolone'),
#         oc_analysis.sel('carboxylic_acid'),
#         oc_analysis.sel('piperazine'),
#         oc_analysis.sel('cyclopropyl')
#     ],
#     group2_sel=oc_analysis.sel('water_oxygen'),
#     center_method='COM',
#     range=(0, 20),
#     force_rerun=False,
#     njobs=1,
#     step=1
# )

In [ ]:
# RDF analysis
# 
rdf_water = oc_analysis.molecular_rdf(
    bin_width=0.1,
    group1_sel=[quinolone,carboxylic_acid, piperazine,cyclopropyl],
    group2_sel=water_oxygen,
    center_method='COM',
    range=(0, 20),
    force_rerun=False,
    njobs=1,
    step=1
)

In [ ]:
# Compare multiple RDFs

plotter.plot_multiple_rdfs(
    rdf_water, 
    xlim=(0, 10),
    show_title=False,
    title_fontsize=22,
    title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
    label_fontsize=22,
    label_fontweight='bold',      # Bold axis labels
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',   # Regular legend text
    xlabel='r (Å)',
    ylabel='g(r)',
    title='CIP moieties-Ow RDFs',
    colors={
        'quinolone': 'red',
        'carboxylic_acid': 'black',
        'piperazine': 'blue',
        'cyclopropyl': 'green'
    },
    legend_ncol=1, 
    legend_framealpha=0.0,
    linewidth=3,
    save_fig=True,
    figsize=(8, 6),
    dpi=600,
    filename='CIP-moieties-Ow-RDFs.png',
)



In [ ]:
# get boundaries using interactive shell editor
# [No selection] > select quinolone
# Selected: quinolone
# [quinolone] > add shell_1 2.0 3.5
# Added shell_1: 2.00 - 3.50 Å
# [quinolone] > add shell_2 3.5 5.5
# Added shell_2: 3.50 - 5.50 Å
# [quinolone] > plot
# [Shows preview plot]
# [quinolone] > save my_boundaries.json
# ✓ Boundaries saved to my_boundaries.json
# [quinolone] > quit

In [ ]:
# boundaries_water = oc_analysis.determine_oc_coordination_shells(
#     rdf_water, 
#     find_peaks_kwargs={'distance': 8, 'height': -2.5, 'prominence': 0.02},
#     plot=True, save_plots=False)

In [ ]:
# print(list(rdf_water.keys()))
boundaries_water = {}

In [ ]:
selected_rdfs = {
    'quinolone': rdf_water['quinolone-Ow'],
    'piperazine': rdf_water['piperazine-Ow'],
    'carboxylic_acid': rdf_water['carboxylic_acid-Ow'],
    'cyclopropyl': rdf_water['cyclopropyl-Ow'],
}

boundaries_water.update(oc_analysis.determine_oc_coordination_shells(
    selected_rdfs,
    find_peaks_kwargs={'distance': 20, 'height': -2.5, 'prominence': 0.005},
    plot=True, save_plots=False))

In [ ]:
# # Refine interactively if needed
# boundaries_water = oc_analysis.interactive_rdf_boundary_editor(
#     rdf_water, initial_boundaries=boundaries_water)

# # Plot final results
# plotter.plot_rdf_with_shells(rdf_water, boundaries_water, ncols=2, save_fig=True, dpi=600)

# boundaries = oc_analysis.interactive_rdf_boundary_editor(rdf_water)

In [ ]:
boundaries_refined = oc_analysis.interactive_rdf_boundary_editor(
    rdf_water,  
    initial_boundaries=boundaries_water
)

In [ ]:
# if boundaries have been refined aready and saved to json, load them here
# Start the editor and load during the session
boundaries_water = oc_analysis.interactive_rdf_boundary_editor(
    rdf_water,
    initial_boundaries={}  # Start empty
)

In [ ]:
# With RCN overlay
rcn_data = oc_analysis.calculate_running_coordination_number(rdf_water)
plotter.plot_rdf_with_shells(
    rdf_water, 
    boundaries_refined, 
    shell_label_fontsize=18, 
    ncols=2,
    show_rcn=True,
    rcn_data=rcn_data,
    rcn_color='red',
    rcn_scale_factor=4.0,
    # Font & text control (NEW DEFAULTS)
    title_fontsize=22,
    title_fontweight='bold',
    show_title=True,
    title_text=None,  # {'quinolone-OW': 'Custom Title'} for overrides
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    # Shell visualization
    shell_alpha=0.4,
    shell_colors=None,  # Auto-generate
    colormap='default',  # 'default' for blue gradient, or 'viridis', 'plasma', etc.
    show_shell_labels=True,
    show_boundary_lines=True,
    boundary_linestyle='--',
    boundary_alpha=0.7,
    boundary_linewidth=1.5,

)

In [ ]:
# RDF between CIP moieties and Na+
# #
rdf_ions = oc_analysis.molecular_rdf(
    bin_width=0.1,
    group1_sel=[quinolone,carboxylic_acid, piperazine,cyclopropyl],
    group2_sel=['name NA', 'name CA'],
    center_method='COM',
    range=(0, 20),
    force_rerun=False,
    njobs=2,
    step=1
)

In [ ]:
plotter.plot_multiple_rdfs(
    rdf_ions,
    xlim=(0, 15),
    title='CIP Moieties - Ion RDFs',
    show_title=False,
    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',
    xlabel='r (Å)',
    ylabel='g(r)',
    colors={
        'quinolone-NA': 'red',          
        'quinolone-CA': 'darkred',
        'piperazine-NA': 'blue',        
        'piperazine-CA': 'darkblue',
        'carboxylic_acid-NA': 'black',
        'carboxylic_acid-CA': 'gray',
        'cyclopropyl-NA': 'green',
        'cyclopropyl-CA': 'darkgreen'
    },
    custom_labels={
        'quinolone-NA': r'Quinolone-Na$^+$',
        'quinolone-CA': r'Quinolone-Ca$^{2+}$',
        'piperazine-NA': r'Piperazine-Na$^+$',
        'piperazine-CA': r'Piperazine-Ca$^{2+}$',
        'carboxylic_acid-NA': r'Carboxylic acid-Na$^+$',
        'carboxylic_acid-CA': r'Carboxylic acid-Ca$^{2+}$',
        'cyclopropyl-NA': r'Cyclopropyl-Na$^+$',
        'cyclopropyl-CA': r'Cyclopropyl-Ca$^{2+}$'
    },
    linestyles=['-', '--', '-', '--', '-', '--', '-', '--'],
    legend_ncol=1, 
    legend_framealpha=0.0,
    linewidth=3,
    save_fig=True,
    figsize=(8, 6),
    dpi=600,
    filename='CIP-moieties-Ion-RDFs.png',
)

In [ ]:
boundaries_water = oc_analysis.determine_oc_coordination_shells(
    rdf_ions, plot=True, plot_range=15, shell_naming='peak', save_plots=True)

In [ ]:
# Initialize boundaries dict for ions
boundaries_ions = {}

In [ ]:
boundaries_ions.update(oc_analysis.determine_oc_coordination_shells(
    rdf_ions,  # Pass entire dict with all ion RDFs
    shell_naming='peak',
    plot_range=15,
    find_peaks_kwargs={'distance': 5, 'height': -1.1, 'prominence': 0.1},
    plot=True, 
    save_plots=False
))

In [ ]:
# Then use interactive boundary editor to refine if needed
boundaries_ions_refined = oc_analysis.interactive_rdf_boundary_editor(
    rdf_ions,
    # initial_boundaries=boundaries_ions
        initial_boundaries={}  # Start empty
)

In [ ]:
# Plot ion RDFs (no RCN)
plotter.plot_rdf_with_shells(
    rdf_ions,  # Use your ion RDF dict
    # boundaries_ions,  # Use your refined ion boundaries
    boundaries_ions_refined,
    shell_naming_style='peak', # beacuse the saved json already has p1 p2 p3 etc
    color_shade_style=None, #, 'modified', # None # 
    shell_label_fontsize=18, 
    ncols=2,
    show_rcn=False,  # Disable RCN for ions
    # Font & text control
    title_fontsize=22,
    title_fontweight='bold',
    show_title=True,
    title_text=None,
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    # Shell visualization
    shell_alpha=0.4,
    shell_colors=None,
    colormap='default',
    show_shell_labels=True,
    show_boundary_lines=True,
    boundary_linestyle='--',
    boundary_alpha=0.7,
    boundary_linewidth=1.5,
    save_fig=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=True,
    show_combined_figure=True,
    save_individual_figures=True,
)

In [ ]:
binding_results = oc_analysis.ion_binding_analysis(
    # target_sel=[quinolone],
    target_sel=[quinolone, piperazine, carboxylic_acid, cyclopropyl],
    ion_types=['NA', 'CA'],
    rdf_boundaries=boundaries_ions_refined,
    calculate_volumes=True, # Enable volume normalization
    peaks={
        'quinolone-NA': ['P2', 'P3', 'P4'],
        'quinolone-CA': ['P2', 'P3', 'P4'],
        'carboxylic_acid-NA': ['P1', 'P2', 'P3'],
        'carboxylic_acid-CA': ['P1', 'P2', 'P3'],
        'piperazine-NA': ['P2', 'P3', 'P4'],
        'piperazine-CA': ['P2', 'P3', 'P4'],
        'cyclopropyl-NA': ['P2','P3'],
        'cyclopropyl-CA': ['P2','P3']        
    },
    save_cache=True,
    force_rerun=False,
    fallback_cutoff=3.5
)

In [ ]:
# binding_results = oc_analysis.ion_binding_analysis(
#     target_sel=[quinolone, carboxylic_acid, piperazine,cyclopropyl],
#     ion_types=['NA', 'K'],
#     cutoff=3.5,
#     step=1,
#     force_rerun=False  # Force rerun since previous had wrong ion_types
# )

In [ ]:
plotter.plot_ion_binding_peak_breakdown(
    binding_results,
    normalize_by_volume=False,  # Enable volume normalization!
    title='CIP Moiety Ion Binding Peak Breakdown',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'cyclopropyl',
    },
    ylabel='Average # of Bound Ions',
    show_title=False,
    
    # Peak control - uses P1/P2/P3/P4 colors automatically
    peak_colors='modified',  # Enhanced color scheme
    peaks_to_show=None,      # Auto-detect available peaks
    
    # Font styling (keeping your settings)
    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',
    legend_sections='full',
    
    # Bar styling
    moiety_hatches={
        'quinolone': '///', 
        'carboxylic_acid': '\\\\\\', 
        'piperazine': 'xxx', 
        'cyclopropyl': 'ooo'
    },
    subplot_layout='horizontal',
    bar_width=0.22,
    
    # Value display
    show_values=True,
    show_peak_values=False,  # Set to True to show individual peak values
    value_fontsize=18,
    
    # Legend styling
    legend_ncol=2, 
    legend_framealpha=0.0,
    
    # Export settings
    filename='ionCIP_binding_peak_breakdown.png',
    save_fig=True,
    figsize=(8, 6),
    dpi=600
)

In [ ]:
plotter.plot_ion_binding_peak_breakdown(
    binding_results,
    normalize_by_volume=True,  # Enable volume normalization!
    title='CIP Moiety Ion Binding Peak Breakdown',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'cyclopropyl',
    },
    ylabel='Average # of Bound Ions',
    show_title=False,
    show_legend=False,
    
    # Peak control - uses P1/P2/P3/P4 colors automatically
    peak_colors='modified',  # Enhanced color scheme
    peaks_to_show=None,      # Auto-detect available peaks
    
    # Font styling (keeping your settings)
    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',
    legend_sections='full',
    
    # Bar styling
    moiety_hatches={
        'quinolone': '///', 
        'carboxylic_acid': '\\\\\\', 
        'piperazine': 'xxx', 
        'cyclopropyl': 'ooo'
    },
    subplot_layout='horizontal',
    bar_width=0.22,
    
    # Value display
    show_values=False,
    show_peak_values=False,  # Set to True to show individual peak values
    value_fontsize=18,
    
    # Legend styling
    legend_ncol=2, 
    legend_framealpha=0.0,
    
    # Export settings
    filename='ionCIP_binding_peak_breakdown.png',
    save_fig=True,
    figsize=(8, 6),
    dpi=600
)

In [ ]:
plotter.plot_ion_binding_comparison(
   binding_results,
    title='CIP Moiety Ion Binding Comparison',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'cyclopropyl',
    },
    ylabel='Average # of Bound Ions',
    show_title=False,
    show_errorbars=True,
    title_fontsize=22,
    title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
    label_fontsize=22,
    label_fontweight='bold',      # Bold axis labels
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',   # Regular legend text

    colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff", 'cyclopropyl': "#00ff00"},
    hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine':'xxx', 'cyclopropyl': 'ooo'},
    # hatches=['///', '\\\\\\', 'xxx'],
    subplot_layout='horizontal',  # Cations left, anions right
    bar_width=0.22,
    show_values=True,
    value_fontsize=18,
    filename='ionCIP_binding.png',
    legend_ncol=1, 
    legend_framealpha=0.0,
    save_fig=True,
    figsize=(8, 6),
    # ylim=(0, 2.25),
    dpi=600
)

In [ ]:
plotter.plot_ion_binding_comparison(
    binding_results,
    normalize_by_volume=True,  # Enable volume normalization!
    volume_calculation_method='weighted_average', # 'sum',
    title='CIP Moiety Ion Binding Comparison',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'cyclopropyl',
    },
    ylabel='Average # of Bound Ions',
    show_title=False,
    show_errorbars=True,
    title_fontsize=22,
    title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
    label_fontsize=22,
    label_fontweight='bold',      # Bold axis labels
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',   # Regular legend text

    colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff", 'cyclopropyl': "#00ff00"},
    hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine':'xxx', 'cyclopropyl': 'ooo'},
    # hatches=['///', '\\\\\\', 'xxx'],
    subplot_layout='horizontal',  # Cations left, anions right
    bar_width=0.22,
    show_values=True,
    value_fontsize=18,
    filename='ionCIP_binding.png',
    legend_ncol=1, 
    legend_framealpha=0.0,
    save_fig=True,
    figsize=(8, 6),
    # ylim=(0, 2.25),
    dpi=600
)

In [ ]:
# 
plotter.plot_ion_binding_timeseries(
    binding_results,
    normalize_by_volume=False,
    density_units='ions/frame/Å³',
    ion_name=['NA','K'],  # Specify which ion
    target_sel=['carboxylic_acid', 'quinolone','piperazine'],
    plot_mode='overlay',
    custom_labels={'quinolone': 'Quinolone', 'carboxylic_acid': 'Carboxylic Acid', 'piperazine': 'Piperazine'},
    colors={'quinolone': 'red', 'carboxylic_acid': 'black', 'piperazine': 'blue'},
    linewidth=2,
    ylabel='Density (ions/frame/Å³)',
    title='Binding Time Series',
    title_fontsize=22,
    title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
    label_fontsize=22,
    label_fontweight='bold',      # Bold axis labels
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',   # Regular legend text
    legend_ncol=2, 
    legend_framealpha=0.0,
    legend_loc='best',
    save_fig=True,
    # ylim=(0, 9),
    filename='binding_timeseries.png',
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=True,
    show_combined_figure=True,
    save_individual_figures=True,
    dpi=600
)

In [ ]:
# 
plotter.plot_ion_binding_timeseries(
    binding_results,
    normalize_by_volume=True,
    volume_calculation_method='sum', # 'weighted_average', #
    # density_units='ions/frame/Å³',
    ion_name=['NA','CA'],  # Specify which ion
    target_sel=['carboxylic_acid', 'quinolone','piperazine'],
    plot_mode='overlay',
    custom_labels={'quinolone': 'Quinolone', 'carboxylic_acid': 'Carboxylic Acid', 'piperazine': 'Piperazine'},
    colors={'quinolone': 'red', 'carboxylic_acid': 'black', 'piperazine': 'blue'},
    linewidth=2,
    # ylabel='Density (ions/frame/Å³)',
    title='Binding Time Series',
    title_fontsize=22,
    title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
    main_ylabel_offset=-0.25,
    label_fontsize=22,
    label_fontweight='bold',      # Bold axis labels
    tick_fontsize=18,
    legend_fontsize=18,
    legend_fontweight='bold',   # Regular legend text
    legend_ncol=2, 
    legend_framealpha=0.0,
    legend_loc='best',
    save_fig=True,
    # ylim=(0, 9),
    filename='binding_timeseries.png',
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=False,
    show_combined_figure=True,
    save_individual_figures=True,
    dpi=600
)

In [ ]:
# # Your desired call now works!
# plotter.plot_ion_selectivity(
#     binding_results,
#     show_title=False,
#     title='CIP Moieties Ion Selectivity (Na⁺ vs K⁺)',
#     title_fontweight='bold',      # 'normal', 'bold', 'light', 'heavy'
#     label_fontweight='bold',      # Bold axis labels
#     legend_fontweight='bold',   # Regular legend text
#     custom_labels={
#         'quinolone': 'Quinolone',
#         'carboxylic_acid': 'Carboxylic Acid',
#         'piperazine': 'Piperazine',
#         'cyclopropyl': 'Cyclopropyl'
#     },
#     colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff", 'cyclopropyl': "#00ff00"},
#     hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine': 'xxx', 'cyclopropyl': 'ooo'},
#     bar_width=0.15,
#     show_values=True,
#     value_fontsize=18,
#     title_fontsize=22,
#     label_fontsize=22,
#     tick_fontsize=18,
#     ylim=(0,0.9),
#     xlabel_rotation=0, 
#     legend_fontsize=18,
#     legend_ncol=1,
#     legend_framealpha=0.0,
#     save_fig=True,
#     filename='Moieties_selectivity.png',
#     dpi=600
# )

In [ ]:
# # Plot using RATIO (default) - more intuitive interpretation
# plotter.plot_ion_selectivity(
#     binding_results,
#     metric='ratio',  # ← NEW parameter! (default)
#     show_title=False,
#     title='CIP Moieties Ion Selectivity - Ratio (K⁺/Na⁺)',
#     title_fontweight='bold',
#     label_fontweight='bold',
#     legend_fontweight='bold',
#     custom_labels={
#         'quinolone': 'Quinolone',
#         'carboxylic_acid': 'Carboxylic Acid',
#         'piperazine': 'Piperazine'
#     },
#     colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff"},
#     hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine': 'xxx'},
#     bar_width=0.15,
#     show_values=True,
#     value_fontsize=18,
#     title_fontsize=22,
#     label_fontsize=22,
#     tick_fontsize=18,
#     ylabel='Selectivity Ratio (K⁺/Na⁺)',
#     xlabel_rotation=0,
#     legend_fontsize=18,
#     legend_ncol=1,
#     legend_framealpha=0.0,
#     save_fig=True,
#     filename='Moieties_selectivity_ratio.png',
#     dpi=600
# )

In [ ]:
# Plot using FRACTION - shows percentage of total binding with peak breakdown
plotter.plot_ion_selectivity_peak_breakdown(
    binding_results,
    # Peak analysis control
    peaks_to_show=None,  # Auto-detect available peaks
    peak_colors='modified',  # Enhanced P1/P2/P3/P4 color scheme
    
    # Selectivity metrics
    metric='ratio',  # ← Shows K/(K+Na) - percentage view
    ion_pair=None,  # Auto-detect ion pair
    
    # Overall plot control
    show_title=False,
    title='CIP Moieties Ion Selectivity Peak Breakdown - Fraction',
    
    # Bar styling
    moiety_hatches={
        'quinolone': '///', 
        'carboxylic_acid': '\\\\\\', 
        'piperazine': 'xxx', 
        'cyclopropyl': 'ooo'
    },
    bar_width=0.15,
    
    # Font styling
    title_fontweight='bold',
    label_fontweight='bold',
    legend_fontweight='bold',
    title_fontsize=22,
    label_fontsize=22,
    tick_fontsize=18,
    legend_fontsize=18,
    
    # Target labels
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'Cyclopropyl'
    },
    
    # Value display
    show_values=True,
    show_peak_values=False,  # Set to True to show individual peak values
    value_fontsize=18,
    
    # Axis control
    ylabel='Selectivity Index',
    # ylabel='Selectivity Fraction \n (K⁺/(K⁺+Na⁺))',
    # ylim=(0, 1),  # Fraction always 0-1
    
    # Legend control
    legend_sections='full',  # Show both peaks and moiety patterns
    legend_ncol=2,
    legend_framealpha=0.0,
    
    # Export settings
    save_fig=True,
    filename='Moieties_selectivity_fraction_peak_breakdown.png',
    dpi=600
)

In [ ]:
# Plot using FRACTION - shows percentage of total binding with peak breakdown
plotter.plot_ion_selectivity_peak_breakdown(
    binding_results,
    normalize_by_volume=True,
    volume_calculation_method='sum', # 'weighted_average', #
    # Peak analysis control
    peaks_to_show=None,  # Auto-detect available peaks
    peak_colors='modified',  # Enhanced P1/P2/P3/P4 color scheme
    
    # Selectivity metrics
    metric='fraction',  # ← Shows K/(K+Na) - percentage view
    ion_pair=None,  # Auto-detect ion pair
    
    main_ylabel_offset=-0.22,  # Adjust if ylabel farto the left
    # Overall plot control
    show_title=False,
    title='CIP Moieties Ion Selectivity Peak Breakdown - Fraction',
    
    # Bar styling
    moiety_hatches={
        'quinolone': '///', 
        'carboxylic_acid': '\\\\\\', 
        'piperazine': 'xxx', 
        'cyclopropyl': 'ooo'
    },
    bar_width=0.15,
    
    # Font styling
    show_legend=False,
    title_fontweight='bold',
    label_fontweight='bold',
    legend_fontweight='bold',
    title_fontsize=22,
    label_fontsize=22,
    tick_fontsize=18,
    legend_fontsize=18,
    
    # Target labels
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'Cyclopropyl'
    },
    
    # Value display
    show_values=True,
    show_peak_values=False,  # Set to True to show individual peak values
    value_fontsize=18,
    
    # Axis control
    ylabel='Selectivity Index',
    # ylabel='Selectivity Fraction \n (K⁺/(K⁺+Na⁺))',
    # ylim=(0, 1),  # Fraction always 0-1
    
    # Legend control
    legend_sections='full',  # Show both peaks and moiety patterns
    legend_ncol=2,
    legend_framealpha=0.0,
    
    # Export settings
    save_fig=True,
    filename='Moieties_selectivity_fraction_peak_breakdown.png',
    dpi=600
)

In [ ]:
# Plot using FRACTION - shows percentage of total binding
plotter.plot_ion_selectivity(
    binding_results,
    metric='fraction',  # ← Shows K/(K+Na) - percentage view
    show_title=False,
    title='CIP Moieties Ion Selectivity - Fraction',
    title_fontweight='bold',
    label_fontweight='bold',
    legend_fontweight='bold',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'Cyclopropyl'
    },
    colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff", 'cyclopropyl': "#00ff00"},
    hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine': 'xxx', 'cyclopropyl': 'ooo'},
    bar_width=0.15,
    show_values=True,
    value_fontsize=18,
    title_fontsize=22,
    label_fontsize=22,
    tick_fontsize=18,
    ylabel='Selectivity Index',
    # ylabel='Selectivity Fraction \n (K⁺/(K⁺+Na⁺))',
    ylim=(0, 1),  # Fraction always 0-1
    xlabel_rotation=0,
    legend_fontsize=18,
    legend_ncol=1,
    legend_framealpha=0.0,
    save_fig=True,
    filename='Moieties_selectivity_fraction.png',
    dpi=600
)

In [ ]:
# Plot using FRACTION - shows percentage of total binding
plotter.plot_ion_selectivity(
    binding_results,
    metric='fraction',  # ← Shows K/(K+Na) - percentage view
    show_title=False,
    title='CIP Moieties Ion Selectivity - Fraction',
    title_fontweight='bold',
    label_fontweight='bold',
    legend_fontweight='bold',
    custom_labels={
        'quinolone': 'Quinolone',
        'carboxylic_acid': 'Carboxylic Acid',
        'piperazine': 'Piperazine',
        'cyclopropyl': 'Cyclopropyl'
    },
    colors={'quinolone': "#ff0303", 'carboxylic_acid': "#000000", 'piperazine': "#0004ff", 'cyclopropyl': "#00ff00"},
    hatches={'quinolone': '///', 'carboxylic_acid': '\\\\\\', 'piperazine': 'xxx', 'cyclopropyl': 'ooo'},
    bar_width=0.15,
    show_values=True,
    value_fontsize=18,
    title_fontsize=22,
    label_fontsize=22,
    tick_fontsize=18,
    ylabel='Selectivity Index',
    # ylabel='Selectivity Fraction \n (K⁺/(K⁺+Na⁺))',
    ylim=(0, 1),  # Fraction always 0-1
    xlabel_rotation=0,
    legend_fontsize=18,
    legend_ncol=1,
    legend_framealpha=0.0,
    save_fig=True,
    filename='Moieties_selectivity_fraction.png',
    dpi=600
)

In [ ]:
# Demonstrate center markers - shows exact binding coordinates
plotter.plot_ion_competition_correlation(
    binding_results,
    normalize_by_volume=True,
    volume_calculation_method='sum', # 'weighted_average', #
    target_sel=['quinolone', 'carboxylic_acid', 'piperazine'],
    ion_pair=('NA', 'MG'),
    plot_type='bubble',
    color_by='frequency',
    colormap='plasma',
    # colormap='YlOrRd',
    bubble_scale=20.0,  # Larger bubbles to show marker utility

    show_bubble_labels=False,

    # Center marker controls
    show_center_markers=False,  # ← Show precise coordinates
    marker_size=25,  # Small dot size
    marker_color='black',  # Black center
    marker_edgecolor='white',  # White halo for visibility
    marker_edgewidth=1.0,  # Thicker edge for clarity
    point_alpha=0.6,
    show_stats=True,
    show_reference_lines=True,
    title='Carboxylic Acid: With Center Markers (Precise Coordinates)',
    title_fontsize=18,
    title_fontweight='bold',
    label_fontsize=14,
    label_fontweight='bold',
    tick_fontsize=12,
    save_fig=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=False,
    show_combined_figure=True,
    save_individual_figures=True,
    filename='CIP_moieties_ion_binding_competition.png',
    show_summary=True,
    dpi=600
)

In [ ]:
# Demonstrate center markers - shows exact binding coordinates
plotter.plot_ion_competition_correlation(
    binding_results,
    normalize_by_volume=True,
    volume_calculation_method='sum', # 'weighted_average', #
    target_sel=['quinolone', 'carboxylic_acid', 'piperazine'],
    ion_pair=('NA', 'MG'),
    plot_type='contour',
    color_by='frequency',
    colormap='plasma',
    # colormap='YlOrRd',
    bubble_scale=20.0,  # Larger bubbles to show marker utility

    show_bubble_labels=False,

    # Center marker controls
    show_center_markers=False,  # ← Show precise coordinates
    marker_size=25,  # Small dot size
    marker_color='black',  # Black center
    marker_edgecolor='white',  # White halo for visibility
    marker_edgewidth=1.0,  # Thicker edge for clarity
    point_alpha=0.6,
    show_stats=True,
    show_reference_lines=True,
    title='Carboxylic Acid: With Center Markers (Precise Coordinates)',
    title_fontsize=18,
    title_fontweight='bold',
    label_fontsize=14,
    label_fontweight='bold',
    tick_fontsize=12,
    save_fig=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=False,
    show_combined_figure=True,
    save_individual_figures=True,
    filename='CIP_moieties_ion_binding_competition.png',
    show_summary=True,
    dpi=600
)

In [ ]:
# Demonstrate center markers - shows exact binding coordinates
plotter.plot_ion_competition_correlation(
    binding_results,
    normalize_by_volume=True,
    volume_calculation_method='sum', # 'weighted_average', #
    target_sel=['quinolone', 'carboxylic_acid', 'piperazine'],
    ion_pair=('NA', 'CA'),

    color_by='density', #'density', #'count', # 'density',# 'frequency',#

    plot_type='contour',
    contour_percentile_range=(0, 100),
    contour_spacing='linear', 
    kde_log_transform=False, 
    colorbar_log_scale=False,


    colormap='plasma',
    normalize_colorbar=True,
    # colorbar_label_style='relative',

    # colormap='YlOrRd',
    bubble_scale=20.0,  # Larger bubbles to show marker utility

    show_bubble_labels=False,

    # Center marker controls
    show_center_markers=False,  # ← Show precise coordinates
    marker_size=25,  # Small dot size
    marker_color='black',  # Black center
    marker_edgecolor='white',  # White halo for visibility
    marker_edgewidth=1.0,  # Thicker edge for clarity
    point_alpha=0.6,
    show_stats=True,
    show_reference_lines=True,
    title='Carboxylic Acid: With Center Markers (Precise Coordinates)',
    title_fontsize=18,
    title_fontweight='bold',
    label_fontsize=14,
    label_fontweight='bold',
    tick_fontsize=12,
    save_fig=True,
    show_individual_figures=False,
    individual_figsize=(8, 6),
    save_combined_figure=False,
    show_combined_figure=True,
    save_individual_figures=True,
    filename='CIP_moieties_ion_binding_competition.png',
    show_summary=True,
    dpi=600
)

In [ ]:
# Using your existing RDF and boundary data
solvation_results = oc_analysis.water_solvation_analysis(
    target_sel=[carboxylic_acid, piperazine, cyclopropyl],
    rdf_data=rdf_water,  # Your existing RDF calculation
    rdf_boundaries=boundaries_refined,  # Your refined boundaries
    solvation_shells={
        'carboxylic_acid-Ow': ['shell_2', 'shell_3'],
        'piperazine-Ow': ['shell_2', 'shell_3'],
        'cyclopropyl-Ow': ['shell_2'],
    },
    include_ions=False,
    calculate_volumes=True,
    force_rerun=False
)

In [ ]:
# Volume-normalized water coordination distribution
plotter.plot_coordination_distribution(
    solvation_results['carboxylic_acid'],
    normalize_by_volume=False,
    volume_calculation_method='sum',
    solvent_type='OW',  # ← Fixed: uppercase OW
    # peak_selection=['shell_2', 'shell_3'],  # total 
    peak_selection=['shell_2'],
    title='Carboxylic Acid Water Solvation',
    show_legend=False,
    show_mean=False,

    # bar_width=0.22, 
    xlim=(-1, 14),
    # x_min=-1,

    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,  # ← Controls axis labels (X/Y axis titles)
    label_fontweight='bold',
    tick_fontsize=18, # ← Controls axis tick numbers
    show_values=False,
    value_label_threshold=0.01,
    value_format='{:.2f}',
    value_label_fontweight='heavy', #'bold',
    value_label_fontsize=14,  # ← Controls bar value labels (numbers above bars)
    # xlabel='Water Density',
    # y_max=0.25,
    # ylim=(0, 0.25),
    save_fig=True
)


In [ ]:
fig, ax = plotter.plot_coordination_distribution(
    solvation_results['carboxylic_acid'],
    normalize_by_volume=False,
    volume_calculation_method='sum',
    solvent_type='OW',
    peak_selection=['shell_2'],
    title='Carboxylic Acid Water Solvation',
    show_legend=False,
    show_mean=False,
    bar_width=1, 
    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    show_values=False,
    value_label_threshold=0.01,
    value_format='{:.2f}',
    value_label_fontweight='heavy',
    value_label_fontsize=14,
    save_fig=False  # Turn off to see debug
)

# Check what the actual x-limits are
print(f"X-axis limits: {ax.get_xlim()}")

In [ ]:
# Multi-site solvation analysis
solvation_sites = {
    'quinolone': quinolone,
    'piperazine': piperazine,
    'carboxylic_acid': carboxylic_acid,
    'cyclopropyl': cyclopropyl
}
solvation_results = oc_analysis.solvation_analysis_organic(
    site_selections=solvation_sites,
    cutoff=3.5,
    step=1,
    include_ions=True,
    save_cache=True,      # Default True
    force_rerun=False     # Default False
)

In [ ]:
# Plot coordination number distribution for water
plotter.plot_coordination_distribution(
    solvation_results['quinolone']['solvent_solvation'],
    title='Water Coordination around Quinolone'
)

# Plot for ions
plotter.plot_coordination_distribution(
    solvation_results['quinolone']['ion_solvation']['NA'],
    title='Na⁺ Coordination around Quinolone'
)

In [ ]:
plotter.plot_coordination_distribution(
    solvation_results['quinolone']['solvent_solvation'],
    title='Water Coordination around Quinolone',
    show_title=False,
    show_legend=False,
    show_mean_line=False,
    align_histograms_to_grids=True,
    title_fontsize=22,
    title_fontweight='bold',
    label_fontsize=22,
    label_fontweight='bold',
    tick_fontsize=18,
    value_label_fontsize=14,
    value_label_threshold=0.01,
    bar_color='navy',
    show_values=True,
    value_format='{:.2f}',
    save_fig=True,
    ylim=(0, 0.25),
    dpi=600
)

In [ ]:
# Print summary directly
for site_name, site_data in solvation_results.items():
    print(f"\n{site_name.upper()}:")
    print(f"  Water coordination: {site_data['solvent_solvation']['average_coordination']:.2f} ± {site_data['solvent_solvation']['coordination_std']:.2f}")
    
    if 'ion_solvation' in site_data:
        print("  Ion coordination:")
        for ion, ion_data in site_data['ion_solvation'].items():
            print(f"    {ion}: {ion_data['average_coordination']:.2f} ± {ion_data['coordination_std']:.2f}")

In [ ]:
solvation_results_Na = oc_analysis.spatial_binding_analysis(
target_sel=[carboxylic_acid, piperazine, quinolone],
ion_type='Na',
rdf_boundaries=boundaries_ions_refined,
peaks={
'quinolone-NA': ['P1', 'P2', 'P3'],
'carboxylic_acid-NA': ['P1', 'P2', 'P3'],
'piperazine-NA': ['P1', 'P2', 'P3'],
},
return_positions=True,
molecular_frame_tracking=True,
force_rerun=False,
method='both'
)

In [ ]:
solvation_results_Ca = oc_analysis.spatial_binding_analysis(
target_sel=[carboxylic_acid, piperazine, quinolone],
ion_type='Ca',
rdf_boundaries=boundaries_ions_refined,
peaks={
'quinolone-CA': ['P1', 'P2', 'P3'],
'carboxylic_acid-CA': ['P1', 'P2', 'P3'],
'piperazine-CA': ['P1', 'P2', 'P3'],
},
return_positions=True,
molecular_frame_tracking=True,
force_rerun=False,
method='both'
)

In [ ]:
plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_Na['carboxylic_acid'],
    structure_file=None,  
    universe=u,
    density_threshold=0.01,
    distance_method='nearest_atom', 
    plot_regions=['P1', 'P2', 'P3'],  # Each region gets full quota
    sphere_size=0.2,
    sphere_opacity=0.8,
    max_spheres=5000,  # Per region, not total!
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Smooth spheres at boundaries
    boundary_sphere_data_extent=False,
    color_shade_style='vibrant',
    boundary_sphere_alpha=0.25,
    show_aromatic_rings=True,
    aromatic_ring_scale=0.8,
    aromatic_ring_color='black',
    aromatic_ring_alpha=0.6,
    aromatic_ring_thickness=0.2,
    use_triangulation=True,
    reconstruction_method='molecular_spherical'# 'com',

)

In [ ]:
plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_Ca['carboxylic_acid'],
    structure_file=None,  
    universe=u,
    density_threshold=0.05,
    distance_method='nearest_atom', 
    plot_regions=['P1', 'P2', 'P3'],  # Each region gets full quota
    sphere_size=0.2,
    sphere_opacity=0.8,
    max_spheres=5000,  # Per region, not total!
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Smooth spheres at boundaries
    boundary_sphere_data_extent=False,
    color_shade_style='vibrant',
    boundary_sphere_alpha=0.25,
    show_aromatic_rings=True,
    aromatic_ring_scale=0.8,
    aromatic_ring_color='black',
    aromatic_ring_alpha=0.6,
    aromatic_ring_thickness=0.2,
    use_triangulation=True,
    reconstruction_method='molecular_spherical'# 'com',

)

In [ ]:
plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_Na['piperazine'],
    structure_file=None,  
    universe=u,
    density_threshold=0.05,
    distance_method='nearest_atom', 
    plot_regions=['P1', 'P2', 'P3'],  # Each region gets full quota
    sphere_size=0.2,
    sphere_opacity=0.8,
    max_spheres=2000,  # Per region, not total!
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Smooth spheres at boundaries
    boundary_sphere_data_extent=False,
    color_shade_style='vibrant',
    boundary_sphere_alpha=0.25,
    show_aromatic_rings=True,
    aromatic_ring_scale=0.8,
    aromatic_ring_color='black',
    aromatic_ring_alpha=0.6,
    aromatic_ring_thickness=0.2,
    use_triangulation=True,
    reconstruction_method='molecular_spherical'# 'com',

)

In [ ]:
plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_Ca['piperazine'],
    structure_file=None,  
    universe=u,
    density_threshold=0.05,
    distance_method='nearest_atom', 
    plot_regions=['P1', 'P2'],  # Each region gets full quota
    sphere_size=0.2,
    sphere_opacity=0.8,
    max_spheres=8000,  # Per region, not total!
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Smooth spheres at boundaries
    boundary_sphere_data_extent=False,
    color_shade_style='vibrant',
    boundary_sphere_alpha=0.25,
    show_aromatic_rings=True,
    aromatic_ring_scale=0.8,
    aromatic_ring_color='black',
    aromatic_ring_alpha=0.6,
    aromatic_ring_thickness=0.2,
    use_triangulation=True,
    reconstruction_method='molecular_spherical'# 'com',

)

In [ ]:
# Analyze spatial binding for K ions
# Use method='both' to get both per-atom and spherical data

spatial_Na = oc_analysis.spatial_binding_analysis(
    target_sel='resname api',
    ion_type='Na',
    cutoff=3.5,
    step=10,  # Use every 10th frame for speed
    method='both',
    angular_bins=(18, 36),  # 10° x 10° angular resolution
    return_positions=True  # Store positions for advanced viz
)

print("\n" + "="*60)
print("Spatial Analysis Results:")
print("="*60)
print(f"Total contacts: {spatial_Na['total_contacts']}")
print(f"Frames analyzed: {spatial_Na['frames_analyzed']}")
print(f"Contact frequency range: {spatial_Na['contact_frequency'].min():.0f} - {spatial_Na['contact_frequency'].max():.0f}")

# Show top 5 hotspot atoms
top_5_idx = np.argsort(spatial_Na['contact_frequency'])[-5:][::-1]
print("\nTop 5 Binding Hotspot Atoms:")
for i, idx in enumerate(top_5_idx, 1):
    atom_name = spatial_Na['atom_names'][idx]
    atom_index = spatial_Na['atom_indices'][idx]
    contacts = spatial_Na['contact_frequency'][idx]
    print(f"  {i}. Atom {atom_name} (index {atom_index}): {contacts:.0f} contacts")

In [ ]:
# Analyze spatial binding for K ions
# Use method='both' to get both per-atom and spherical data

spatial_K = oc_analysis.spatial_binding_analysis(
    target_sel='resname api',
    ion_type='K',
    cutoff=3.5,
    step=10,  # Use every 10th frame for speed
    method='both',
    angular_bins=(18, 36),  # 10° x 10° angular resolution
    return_positions=True  # Store positions for advanced viz
)

print("\n" + "="*60)
print("Spatial Analysis Results:")
print("="*60)
print(f"Total contacts: {spatial_K['total_contacts']}")
print(f"Frames analyzed: {spatial_K['frames_analyzed']}")
print(f"Contact frequency range: {spatial_K['contact_frequency'].min():.0f} - {spatial_K['contact_frequency'].max():.0f}")

# Show top 5 hotspot atoms
top_5_idx = np.argsort(spatial_K['contact_frequency'])[-5:][::-1]
print("\nTop 5 Binding Hotspot Atoms:")
for i, idx in enumerate(top_5_idx, 1):
    atom_name = spatial_K['atom_names'][idx]
    atom_index = spatial_K['atom_indices'][idx]
    contacts = spatial_K['contact_frequency'][idx]
    print(f"  {i}. Atom {atom_name} (index {atom_index}): {contacts:.0f} contacts")

In [ ]:
# Check K+ hotspot atom positions
print("="*70)
print("K+ BINDING HOTSPOT VERIFICATION")
print("="*70)

# Get top 3 hotspot atoms
top_3_idx = np.argsort(spatial_K['contact_frequency'])[-3:][::-1]

print("\nTop 3 Hotspot Atoms:")
for i, idx in enumerate(top_3_idx, 1):
    atom_name = spatial_K['atom_names'][idx]
    atom_index = spatial_K['atom_indices'][idx]
    contacts = spatial_K['contact_frequency'][idx]
    position = spatial_K['atom_positions'][idx]
    
    print(f"\n  {i}. Atom: {atom_name} (index {atom_index})")
    print(f"     Contacts: {contacts:.0f}")
    print(f"     Position: X={position[0]:6.2f}, Y={position[1]:6.2f}, Z={position[2]:6.2f} Å")

print("\n" + "="*70)
print("WHAT TO CHECK:")
print("="*70)
print("After running visualization with 'CIP_frame0.pdb':")
print("  1. Red spheres (high density) should appear near the positions above")
print(f"  2. Specifically, look for red spheres near atom {spatial_K['atom_names'][top_3_idx[0]]}")
print("  3. Check if the functional group matches:")
print("     - O1, O3 = carboxyl end (COO⁻, should bind cations)")
print("     - N13, N16 = piperazine end (less negative)")
print("\nIf red spheres are at carboxyl end → ✓ CORRECT!")
print("If red spheres are at piperazine end → ✗ STILL FLIPPED")
print("="*70)

### Test Visualization with Frame 0 PDB

Now test with the correctly oriented PDB structure.

## PyMOL Visualization in Notebook

Visualize directly in Jupyter using PyMOL's Python API.

In [ ]:
# Analyze Cl- binding for comparison
spatial_Cl = oc_analysis.spatial_binding_analysis(
    target_sel='resname api',
    ion_type='CL',  # Now analyze chloride
    cutoff=3.5,
    step=10,
    method='both',
    angular_bins=(18, 36)
)

# Plot side-by-side comparison
fig = plt.figure(figsize=(16, 6))

# K+ on left
ax1 = fig.add_subplot(121, projection='3d')
positions_K = spatial_K['atom_positions']
freq_K = spatial_K['contact_frequency']
scatter1 = ax1.scatter(positions_K[:, 0], positions_K[:, 1], positions_K[:, 2],
                      c=freq_K, cmap='Blues', s=100 + 400 * freq_K / freq_K.max(),
                      alpha=0.7, edgecolors='black', linewidths=0.5)
ax1.set_title('K+ Binding Sites', fontsize=14, fontweight='bold')
ax1.set_xlabel('X (Å)')
ax1.set_ylabel('Y (Å)')
ax1.set_zlabel('Z (Å)')
plt.colorbar(scatter1, ax=ax1, shrink=0.7, label='Contacts')

# Cl- on right
ax2 = fig.add_subplot(122, projection='3d')
positions_Cl = spatial_Cl['atom_positions']
freq_Cl = spatial_Cl['contact_frequency']
scatter2 = ax2.scatter(positions_Cl[:, 0], positions_Cl[:, 1], positions_Cl[:, 2],
                      c=freq_Cl, cmap='Reds', s=100 + 400 * freq_Cl / freq_Cl.max(),
                      alpha=0.7, edgecolors='black', linewidths=0.5)
ax2.set_title('Cl- Binding Sites', fontsize=14, fontweight='bold')
ax2.set_xlabel('X (Å)')
ax2.set_ylabel('Y (Å)')
ax2.set_zlabel('Z (Å)')
plt.colorbar(scatter2, ax=ax2, shrink=0.7, label='Contacts')

fig.suptitle('Ion Binding Site Comparison', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"\nK+ binding: {spatial_K['total_contacts']} total contacts")
print(f"Cl- binding: {spatial_Cl['total_contacts']} total contacts")

In [ ]:
# Interactive 3D visualization: CIP molecule + K+ ion binding positions in space
# Molecule: First frame structure (preserves geometry), centered at origin
# Ions: Spatial distribution across all frames, relative to COM
plotter.plot_spatial_binding_interactive(
    spatial_results=spatial_K,
    structure_file='CIP.pdb',
    universe=u,  # For getting first frame structure
    density_threshold=0.02,  # Show positions with >2% of max density
    sphere_size=0.4,  # Size of ion position spheres
    sphere_opacity=0.3,  # 70% transparent
    stick_radius=0.15,  # Thinner sticks for molecule
    ball_scale=0.3,  # Ball size for molecule atoms
    width=800,
    height=600,
    show_output=True,
    max_spheres=500  # Limit for performance
)

In [ ]:
competition = oc_analysis.ion_competition_analysis(
    target_sel='resname api',
    cutoff=5.0,
    step=1
)

In [ ]:
pair_results = oc_analysis.specific_ion_pair_analysis(
    cation_sel='name K',
    anion_sel='name CL',
    cutoff=4.0,
    step=1
)

In [ ]:
selectivity = oc_analysis.ion_selectivity_analysis(
    target_sel='resname api',
    reference_distance=5.0,
    step=1
)

In [ ]:
solvation = oc_analysis.solvation_analysis_organic(
    site_selections={
        'quinolone': quinolone,
        'piperazine': piperazine,
        'carboxylic': carboxylic_acid
    },
    cutoff=3.5,
    step=1,
    include_ions=True
)

In [ ]:
condensation = oc_analysis.counterion_condensation_analysis(
    polymer_sel='resname api',
    charged_sites_sel='name O* or name N*',  # Charged sites
    cutoff=3.5,
    step=1
)

In [ ]:
# 1. Calculate RDFs (already done)
# 2. Ion binding to CIP
binding = oc_analysis.ion_binding_analysis('resname api', cutoff=3.5)

# 3. Solvation of molecular parts
solvation = oc_analysis.solvation_analysis_organic({
    'quinolone': quinolone,
    'piperazine': piperazine
}, include_ions=True)

# 4. Ion pair analysis
pairs = oc_analysis.specific_ion_pair_analysis('name K', 'name CL')

# 5. Generate summary
oc_analysis.summary_report()

In [ ]:
# Debug the solvation_results structure
print("🔍 DEBUGGING SOLVATION RESULTS STRUCTURE:")
print("\n1. Top-level keys in solvation_results:")
for key in solvation_results.keys():
    print(f"   {key}")

print(f"\n2. Structure of carboxylic_acid results:")
carb_data = solvation_results['carboxylic_acid']
for key in carb_data.keys():
    print(f"   {key}")

print(f"\n3. Water solvation data structure:")
if 'water_solvation' in carb_data:
    water_data = carb_data['water_solvation']
    print(f"   Solvent types: {list(water_data.keys())}")
    
    # Check what's in the first solvent type
    first_solvent = list(water_data.keys())[0]
    solvent_data = water_data[first_solvent]
    print(f"   Structure of '{first_solvent}': {list(solvent_data.keys())}")
    
    # Check peak analysis
    if 'peak_analysis' in solvent_data:
        peaks = solvent_data['peak_analysis']
        print(f"   Available peaks/shells: {list(peaks.keys())}")
        
        # Show structure of first peak
        if len(peaks) > 0:
            first_peak = list(peaks.keys())[0]
            peak_data = peaks[first_peak]
            print(f"   Structure of '{first_peak}': {list(peak_data.keys())}")
else:
    print("   No 'water_solvation' key found!")

In [ ]:
# Test 1: Fixed discrete region filtering for P2 + P3
print("=== TESTING DISCRETE REGION FILTERING (P2 + P3) ===")

plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_fixed['carboxylic_acid'],
    structure_file=None,  
    universe=u,
    density_threshold=0.01,
    distance_method='nearest_atom', 
    plot_regions=['P2', 'P3'],  # Should now show discrete P2 AND P3 regions
    sphere_size=0.2,
    sphere_opacity=0.4,
    max_spheres=1000,  # Increased to see both regions
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=False,  # Test without boundaries first
)

In [ ]:
# Test 2: With boundary spheres enabled  
print("=== TESTING WITH BOUNDARY SPHERES ===")

plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_fixed['carboxylic_acid'],
    structure_file=None,  
    universe=u,
    density_threshold=0.01,
    distance_method='nearest_atom', 
    plot_regions=['P2', 'P3'],  
    sphere_size=0.2,
    sphere_opacity=0.4,
    max_spheres=2000,  
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Enable boundary visualization
)

In [ ]:
# Test: Fixed per-region subsampling + smooth boundary spheres
print("=== TESTING FINAL FIXES ===")
print("Expected: P2 gets 1000 spheres + P3 gets 1000 spheres = ~2000 total")
print("Expected: Smooth transparent boundary spheres at 3.9 Å (P2) and 6.0 Å (P3)")

plotter.plot_spatial_binding_interactive(
    spatial_results=solvation_results_fixed['carboxylic_acid'],
    structure_file=None,  
    universe=u,
    density_threshold=0.01,
    distance_method='nearest_atom', 
    plot_regions=['P1', 'P2', 'P3'],  # Each region gets full quota
    sphere_size=0.2,
    sphere_opacity=0.4,
    max_spheres=1000,  # Per region, not total!
    stick_radius=0.15,
    ball_scale=0.3,
    width=800,
    height=600,
    show_output=True,
    show_boundary_spheres=True,  # Smooth spheres at boundaries
    boundary_sphere_data_extent=False,
    color_shade_style='vibrant',
    boundary_sphere_alpha=0.6
)